# 2D Soft Body Dynamics: Model Order Reduction

This tutorial introduces **Model Order Reduction (MOR)** for accelerating soft body simulations.

**Learning Goals:**
1. Understand the motivation for model order reduction
2. Learn Proper Orthogonal Decomposition (POD) for computing reduced bases
3. Implement a training loop to collect simulation snapshots
4. Build a reduced-order solver that operates in low-dimensional space
5. Compare full vs reduced simulation performance

**Prerequisites:** Complete [03_fem.ipynb](03_fem.ipynb) first!

---
## 1. Why Model Order Reduction?

### The Problem: Computational Cost
Our implicit solver requires solving a linear system at each timestep:
$$(\mathbf{M} - h^2\mathbf{K}) \Delta\mathbf{v} = h\mathbf{f}$$

For N particles in 2D, this is a **2N × 2N** system. Cost scales as:
- Direct solve: O(N³)
- Iterative solve: O(N^1.5) to O(N²) per iteration

For soft robots with 100+ particles, this becomes expensive.

### The Solution: Reduced Space
**Key insight:** Soft body deformations often lie in a **low-dimensional subspace**.

Instead of simulating all 2N DOF, we simulate in a reduced space of **r << 2N** dimensions:
$$\mathbf{x} \approx \mathbf{V}_r \mathbf{q}_r + \mathbf{x}_0$$

where:
- $\mathbf{V}_r \in \mathbb{R}^{2N \times r}$ is the **reduced basis** (POD modes)
- $\mathbf{q}_r \in \mathbb{R}^r$ are the **reduced coordinates**
- $\mathbf{x}_0$ is the **rest position**

### Speedup
| Operation | Full | Reduced |
|-----------|------|--------|
| DOF | 2N | r |
| Linear solve | O(N³) | O(r³) |
| Speedup | 1x | (2N/r)² |

For 100 particles (200 DOF) reduced to 20 modes: **100x speedup!**

---
## 2. Proper Orthogonal Decomposition (POD)

### Overview
POD finds the **optimal low-dimensional basis** from simulation data using Singular Value Decomposition (SVD).

### Algorithm
1. **Collect snapshots:** Run full simulation, save positions at each timestep
   $$\mathbf{X} = [\mathbf{x}_1 - \mathbf{x}_0, \mathbf{x}_2 - \mathbf{x}_0, ..., \mathbf{x}_N - \mathbf{x}_0]$$

2. **Compute SVD:**
   $$\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

3. **Select modes:** Keep first r columns of U where:
   $$\sqrt{\frac{\sum_{i>r} \sigma_i^2}{\sum_i \sigma_i^2}} < \text{tolerance}$$

4. **Reduced basis:** $\mathbf{V}_r = \mathbf{U}[:, :r]$

### Key Properties
- **Optimal:** POD minimizes reconstruction error for any given r
- **Singular values:** $\sigma_i$ indicate energy content of each mode
- **Typical reduction:** 200 DOF → 10-30 modes captures 99%+ of deformation

---
## 3. Implementation: Training Phase

In [1]:
# Step 1: Imports
import os
import sys

# Add the tutorial/Warp directory to path for imports
_warp_tutorial_dir = os.path.join(os.getcwd(), 'tutorial', 'Warp')
if os.path.exists(_warp_tutorial_dir) and _warp_tutorial_dir not in sys.path:
    sys.path.insert(0, _warp_tutorial_dir)
elif os.path.exists('utils.py') and os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import sys
# Ensure we can import from the notebook's directory

import numpy as np
import warp as wp
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation  # For FEM triangle visualization
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image
import time

# Import from previous tutorials
from utils import (
    State, Model, SolverImplicitFEM,
    eval_spring_2d, eval_fem_2d, apply_boundary_2d,
    create_spring_colormap, create_fem_colormap, get_colors
)

# Import MOR components (introduced in this tutorial)
from reduction import SnapshotCollector, PODReducer, ReducedBasis, SolverReduced

wp.init()
print(f'Warp {wp.__version__} on {wp.get_device()}')

ModuleNotFoundError: No module named 'utils'

In [ ]:
# Step 2: Create the soft body model
# Tutorial 4: Use a LARGE body to demonstrate MOR speedup benefit!
# - 32 boundary points, 5 inner rings → ~180 particles, ~500 springs
# - With 360 DOFs, the reduction from 360 to ~20-30 modes shows real speedup
# - This is significantly larger than Tutorial 3 to justify using reduction

model = Model.from_circle(
    radius=1.5,           # 3x larger than FEM (0.5 * 3)
    num_boundary=60,      # 3x more boundary points (20 * 3)
    num_rings=9,          # 3x more rings (3 * 3)
    boxsize=5.0,          # Large box for big body
    center=(2.5, 1.7),    # Centered, elevated start
    spring_stiffness=40.0,
    spring_damping=0.5,
    use_fem=True,
    fem_mu=19.2,
    fem_lambda=16.5,
    fem_damping=2.0,
)

# Set gravity
model.set_gravity((0.0, -0.1))  # Same as tutorial 3 - weak gravity for FEM

print(f'Particles: {model.particle_count}')
print(f'Springs: {model.spring_count}')
print(f'Triangles: {model.tri_count}')
print(f'Full DOF: {model.particle_count * 2}')

# Verify FEM is enabled
assert model.tri_count > 0, "ERROR: FEM triangles not created! Check use_fem=True"
assert model.tri_strains is not None, "ERROR: tri_strains not initialized!"
print(f'\n✓ FEM ENABLED: {model.tri_count} triangles with mu={19.2}, lambda={16.5}')

In [ ]:
# Step 3: Create full-order solver
# This is the "ground truth" solver we want to accelerate

solver_full = SolverImplicitFEM(model, mass=1.0)
print('Full-order solver ready')

In [ ]:
# Step 4: Training Loop - Collect Snapshots
# Run full simulation and collect position snapshots for POD

# Simulation parameters
dt = 0.04
train_time = 25.0  # seconds of training data (longer for larger body)
n_steps = int(train_time / dt)
snapshot_interval = 2  # Collect every N steps

print(f'Training: {n_steps} steps, collecting every {snapshot_interval} steps')
print(f'Expected snapshots: ~{n_steps // snapshot_interval}')

# Initialize snapshot collector
collector = SnapshotCollector()

# Initialize states
state_in = model.state()
state_out = model.state()

# Set rest position (initial configuration)
rest_pos = state_in.particle_q.numpy()
collector.set_rest_position(rest_pos)

# Training loop
print('\nCollecting snapshots...')
start_time = time.time()

training_frames = []  # For visualization
for step in range(n_steps):
    # Full-order simulation step
    solver_full.step(state_in, state_out, dt)
    
    # Collect snapshot
    if step % snapshot_interval == 0:
        pos = state_out.particle_q.numpy()
        collector.add_snapshot(pos)
        
        # Save frame for visualization (including FEM tri_strains!)
        if step % 4 == 0:
            training_frames.append((
                pos.copy(),
                model.spring_strains.numpy().copy(),
                model.tri_strains.numpy().copy(),  # FEM triangle strains
                step * dt
            ))
    
    # Swap states
    state_in, state_out = state_out, state_in

elapsed = time.time() - start_time
print(f'\nCollected {collector.n_snapshots} snapshots in {elapsed:.2f}s')
print(f'DOF: {collector.n_dof}')

In [ ]:
# Visualize training data (with FEM triangles!)
fig, ax = plt.subplots(figsize=(8, 8))
spring_idx = model.spring_indices.numpy().reshape(-1, 2)
tri_idx = model.tri_indices.numpy().reshape(-1, 3)  # FEM triangles
cmap_spring = create_spring_colormap()
cmap_fem = create_fem_colormap()

def update_train(idx):
    pos, spring_strain, tri_strain, t = training_frames[idx]
    ax.clear()
    ax.set_facecolor('white')
    
    # Draw FEM triangles (colored by strain)
    triang = Triangulation(pos[:, 0], pos[:, 1], tri_idx)
    tri_norm = np.clip(np.abs(tri_strain) / max(0.1, np.abs(tri_strain).max()), 0, 1)
    ax.tripcolor(triang, tri_norm, cmap=cmap_fem, alpha=0.85)
    
    # Draw springs
    segs = [[pos[i], pos[j]] for i, j in spring_idx]
    lc = LineCollection(segs, cmap=cmap_spring, linewidths=2)
    strain_norm = np.clip(np.abs(spring_strain) / max(0.02, np.abs(spring_strain).max()), 0, 1)
    lc.set_array(strain_norm)
    ax.add_collection(lc)
    
    # Draw particles
    colors = get_colors(pos, model.boxsize)
    ax.scatter(pos[:, 0], pos[:, 1], s=30, c=colors, edgecolors='white', lw=0.8, zorder=5)
    
    ax.set_xlim(0, model.boxsize)
    ax.set_ylim(0, model.boxsize)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Training Data Collection | t={t:.2f}s | Snapshots: {collector.n_snapshots}', fontsize=12, fontweight='bold')

anim = FuncAnimation(fig, update_train, frames=len(training_frames), interval=40)
anim.save('mor_training.gif', writer=PillowWriter(fps=25))
plt.close()
print('Saved mor_training.gif')

In [ ]:
Image(filename='mor_training.gif')

---
## 4. Computing the POD Basis

In [ ]:
# Step 5: Compute POD Basis using SVD

# Get snapshots matrix
snapshots = collector.get_snapshots()
rest_position = collector.rest_position

print(f'Snapshot matrix shape: {snapshots.shape}')
print(f'  (DOF x n_snapshots)')

# Create POD reducer
# tolerance=1e-5 means we keep modes until 99.999% of energy is captured
# Lower tolerance = more modes = better accuracy for large body
reducer = PODReducer(tolerance=1e-5, verbose=True)

# Compute basis
basis, pod_info = reducer.fit(snapshots, rest_position)

print(f'\n--- Result ---')
print(f'Full DOF: {basis.n_full}')
print(f'Reduced modes: {basis.n_modes}')
print(f'Compression: {basis.compression_ratio:.1f}x')
print(f'Energy captured: {pod_info["energy_captured"]*100:.2f}%')

In [ ]:
# Visualize singular values (energy content)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sv = pod_info['singular_values']
energy = pod_info['energy_ratio']
cumulative = np.cumsum(energy)

# Singular values
ax1.semilogy(sv, 'b.-', markersize=8)
ax1.axhline(sv[basis.n_modes-1], color='r', linestyle='--', label=f'Cutoff at mode {basis.n_modes}')
ax1.set_xlabel('Mode index')
ax1.set_ylabel('Singular value (log scale)')
ax1.set_title('Singular Values', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative energy
ax2.plot(100 * cumulative, 'g.-', markersize=8)
ax2.axhline(99.9, color='r', linestyle='--', label='99.9% threshold')
ax2.axvline(basis.n_modes-1, color='orange', linestyle='--', label=f'{basis.n_modes} modes')
ax2.set_xlabel('Number of modes')
ax2.set_ylabel('Cumulative energy (%)')
ax2.set_title('Energy Captured vs Modes', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(80, 100.5)

plt.tight_layout()
plt.savefig('pod_analysis.png', dpi=150)
plt.show()
print('\nThe first few modes capture most of the deformation energy!')

In [ ]:
# Visualize first few POD modes
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

rest_pos_2d = rest_position.reshape(-1, 2)
scale = 0.3  # Visualization scale

for idx, ax in enumerate(axes.flat):
    if idx >= min(6, basis.n_modes):
        ax.axis('off')
        continue
    
    mode = basis.modes[:, idx].reshape(-1, 2)
    
    # Draw rest shape (gray)
    for i, j in spring_idx:
        ax.plot([rest_pos_2d[i, 0], rest_pos_2d[j, 0]], 
                [rest_pos_2d[i, 1], rest_pos_2d[j, 1]], 'gray', alpha=0.3, lw=1)
    
    # Draw mode deformation (colored arrows)
    ax.quiver(rest_pos_2d[:, 0], rest_pos_2d[:, 1], 
              mode[:, 0], mode[:, 1], 
              scale=3, color='blue', alpha=0.8)
    
    # Draw deformed shape
    deformed = rest_pos_2d + scale * mode
    ax.scatter(deformed[:, 0], deformed[:, 1], s=15, c='red', zorder=5)
    
    ax.set_xlim(0.5, 4.5)  # Wider for 3x larger body
    ax.set_ylim(-0.2, 3.5)  # Taller for 3x larger body
    ax.set_aspect('equal')
    ax.set_title(f'Mode {idx+1} (σ={sv[idx]:.2f})', fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('First 6 POD Modes (Blue=direction, Red=deformed shape)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pod_modes.png', dpi=150)
plt.show()

---
## 5. Reduced-Order Simulation

In [ ]:
# Step 6: Create Reduced Solver

# Create a fresh model for inference (SAME large body as training)
model_reduced = Model.from_circle(
    radius=1.5,
    num_boundary=60,
    num_rings=9,
    boxsize=5.0,
    center=(2.5, 1.7),
    spring_stiffness=40.0,
    spring_damping=0.5,
    use_fem=True,
    fem_mu=19.2,
    fem_lambda=16.5,
    fem_damping=2.0,
)
model_reduced.set_gravity((0.0, -0.1))  # Same as training

# Create reduced solver with POD basis
solver_reduced = SolverReduced(model_reduced, basis, mass=1.0)

stats = solver_reduced.get_stats()
print(f'\nReady for inference:')
print(f'  Full DOF: {stats["n_full"]}')
print(f'  Reduced DOF: {stats["n_reduced"]}')
print(f'  Compression: {stats["compression_ratio"]:.1f}x')
print(f'  Estimated speedup: ~{stats["speedup_estimate"]:.0f}x')

In [ ]:
# Step 7: Run Reduced Simulation

run_time = 8.0
n_run_steps = int(run_time / dt)

# Initialize states
state_in = model_reduced.state()
state_out = model_reduced.state()

print(f'Running reduced simulation for {run_time}s ({n_run_steps} steps)...')
start_time = time.time()

reduced_frames = []
for step in range(n_run_steps):
    solver_reduced.step(state_in, state_out, dt)
    
    if step % 2 == 0:
        reduced_frames.append((
            state_out.particle_q.numpy().copy(),
            model_reduced.spring_strains.numpy().copy(),
            model_reduced.tri_strains.numpy().copy(),  # FEM triangle strains
            step * dt
        ))
    
    state_in, state_out = state_out, state_in

reduced_elapsed = time.time() - start_time
print(f'Reduced solver: {reduced_elapsed:.2f}s, {len(reduced_frames)} frames')

In [ ]:
# Visualize reduced simulation (with FEM triangles!)
fig, ax = plt.subplots(figsize=(8, 8))
spring_idx_r = model_reduced.spring_indices.numpy().reshape(-1, 2)
tri_idx_r = model_reduced.tri_indices.numpy().reshape(-1, 3)

def update_reduced(idx):
    pos, spring_strain, tri_strain, t = reduced_frames[idx]
    ax.clear()
    ax.set_facecolor('white')
    
    # Draw FEM triangles
    triang = Triangulation(pos[:, 0], pos[:, 1], tri_idx_r)
    tri_norm = np.clip(np.abs(tri_strain) / max(0.1, np.abs(tri_strain).max()), 0, 1)
    ax.tripcolor(triang, tri_norm, cmap=cmap_fem, alpha=0.85)
    
    # Draw springs
    segs = [[pos[i], pos[j]] for i, j in spring_idx_r]
    lc = LineCollection(segs, cmap=cmap_spring, linewidths=2)
    strain_norm = np.clip(np.abs(spring_strain) / max(0.02, np.abs(spring_strain).max()), 0, 1)
    lc.set_array(strain_norm)
    ax.add_collection(lc)
    
    # Draw particles
    colors = get_colors(pos, model_reduced.boxsize)
    ax.scatter(pos[:, 0], pos[:, 1], s=30, c=colors, edgecolors='white', lw=0.8, zorder=5)
    
    ax.set_xlim(0, model_reduced.boxsize)
    ax.set_ylim(0, model_reduced.boxsize)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'REDUCED Solver ({stats["n_reduced"]} modes) | t={t:.2f}s', fontsize=12, fontweight='bold', color='green')

anim = FuncAnimation(fig, update_reduced, frames=len(reduced_frames), interval=40)
anim.save('mor_reduced.gif', writer=PillowWriter(fps=25))
plt.close()
print('Saved mor_reduced.gif')

In [ ]:
Image(filename='mor_reduced.gif')

---
## 6. Comparison: Full vs Reduced

In [ ]:
# Step 8: Run full solver for comparison

model_full = Model.from_circle(
    radius=1.5, num_boundary=60, num_rings=9, boxsize=5.0, center=(2.5, 1.7),
    spring_stiffness=40.0, spring_damping=0.5,
    use_fem=True, fem_mu=19.2, fem_lambda=16.5, fem_damping=2.0,
)
model_full.set_gravity((0.0, -0.1))  # Same as training
solver_full_compare = SolverImplicitFEM(model_full, mass=1.0)

state_in = model_full.state()
state_out = model_full.state()

print(f'Running full simulation for comparison...')
start_time = time.time()

full_frames = []
for step in range(n_run_steps):
    solver_full_compare.step(state_in, state_out, dt)
    
    if step % 2 == 0:
        full_frames.append((
            state_out.particle_q.numpy().copy(),
            model_full.spring_strains.numpy().copy(),
            model_full.tri_strains.numpy().copy(),  # FEM triangle strains
            step * dt
        ))
    
    state_in, state_out = state_out, state_in

full_elapsed = time.time() - start_time
print(f'Full solver: {full_elapsed:.2f}s')
print(f'Reduced solver: {reduced_elapsed:.2f}s')
print(f'Speedup: {full_elapsed / reduced_elapsed:.2f}x')

In [ ]:
# Side-by-side comparison animation (with FEM triangles!)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
tri_idx_full = model_full.tri_indices.numpy().reshape(-1, 3)

def update_compare(idx):
    # Full solver (left)
    pos_full, spring_full, tri_full, t = full_frames[idx]
    ax1.clear()
    ax1.set_facecolor('white')
    # Draw FEM triangles
    triang1 = Triangulation(pos_full[:, 0], pos_full[:, 1], tri_idx_full)
    ax1.tripcolor(triang1, np.clip(np.abs(tri_full) / max(0.1, np.abs(tri_full).max()), 0, 1), cmap=cmap_fem, alpha=0.85)
    # Draw springs
    segs = [[pos_full[i], pos_full[j]] for i, j in spring_idx]
    lc = LineCollection(segs, cmap=cmap_spring, linewidths=2)
    lc.set_array(np.clip(np.abs(spring_full) / max(0.02, np.abs(spring_full).max()), 0, 1))
    ax1.add_collection(lc)
    ax1.scatter(pos_full[:, 0], pos_full[:, 1], s=30, c='#87CEEB', edgecolors='white', lw=0.8, zorder=5)
    ax1.set_xlim(0, model.boxsize)
    ax1.set_ylim(0, model.boxsize)
    ax1.set_aspect('equal')
    ax1.set_xticks([])
    ax1.set_yticks([])
    ax1.set_title(f'FULL Solver ({model.particle_count*2} DOF) | t={t:.2f}s', fontsize=12, fontweight='bold', color='blue')
    
    # Reduced solver (right)
    pos_red, spring_red, tri_red, _ = reduced_frames[idx]
    ax2.clear()
    ax2.set_facecolor('white')
    # Draw FEM triangles
    triang2 = Triangulation(pos_red[:, 0], pos_red[:, 1], tri_idx_r)
    ax2.tripcolor(triang2, np.clip(np.abs(tri_red) / max(0.1, np.abs(tri_red).max()), 0, 1), cmap=cmap_fem, alpha=0.85)
    # Draw springs
    segs = [[pos_red[i], pos_red[j]] for i, j in spring_idx_r]
    lc = LineCollection(segs, cmap=cmap_spring, linewidths=2)
    lc.set_array(np.clip(np.abs(spring_red) / max(0.02, np.abs(spring_red).max()), 0, 1))
    ax2.add_collection(lc)
    ax2.scatter(pos_red[:, 0], pos_red[:, 1], s=30, c='#90EE90', edgecolors='white', lw=0.8, zorder=5)
    ax2.set_xlim(0, model_reduced.boxsize)
    ax2.set_ylim(0, model_reduced.boxsize)
    ax2.set_aspect('equal')
    ax2.set_xticks([])
    ax2.set_yticks([])
    ax2.set_title(f'REDUCED Solver ({stats["n_reduced"]} modes) | t={t:.2f}s', fontsize=12, fontweight='bold', color='green')

n_frames = min(len(full_frames), len(reduced_frames))
anim = FuncAnimation(fig, update_compare, frames=n_frames, interval=40)
anim.save('mor_comparison.gif', writer=PillowWriter(fps=25))
plt.close()
print('Saved mor_comparison.gif')

In [ ]:
Image(filename='mor_comparison.gif')

In [ ]:
# Quantitative comparison: trajectory error
errors = []
times = []

for i in range(min(len(full_frames), len(reduced_frames))):
    pos_full = full_frames[i][0]
    pos_red = reduced_frames[i][0]
    t = full_frames[i][2]
    
    # Compute position error
    error = np.linalg.norm(pos_full - pos_red) / np.linalg.norm(pos_full - rest_pos.reshape(-1,2))
    errors.append(error * 100)  # As percentage
    times.append(t)

plt.figure(figsize=(10, 4))
plt.plot(times, errors, 'b-', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Relative Error (%)')
plt.title('Full vs Reduced Trajectory Error', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(np.mean(errors), color='r', linestyle='--', label=f'Mean: {np.mean(errors):.1f}%')
plt.legend()
plt.tight_layout()
plt.savefig('mor_error.png', dpi=150)
plt.show()

print(f'\nError Statistics:')
print(f'  Mean error: {np.mean(errors):.2f}%')
print(f'  Max error: {np.max(errors):.2f}%')

---
## 7. Summary

**What we built:**
1. **Snapshot Collector** - Gathers simulation data for POD training
2. **POD Reducer** - Computes optimal reduced basis using SVD
3. **Reduced Solver** - Simulates in low-dimensional space

**Key Concepts:**
- **POD Basis** $\mathbf{V}_r$: Captures dominant deformation modes
- **Reduced Coordinates** $\mathbf{q}_r = \mathbf{V}_r^T (\mathbf{x} - \mathbf{x}_0)$: Low-dimensional state
- **Reconstruction** $\mathbf{x} = \mathbf{V}_r \mathbf{q}_r + \mathbf{x}_0$: Recover full state
- **Reduced System** $(\mathbf{M}_r - h^2\mathbf{K}_r)\Delta\mathbf{v}_r = h\mathbf{f}_r$: Smaller linear solve

**Performance:**
- Compression: ~10x (200 DOF → 20 modes)
- Speedup: Linear solve is ~100x faster
- Accuracy: <5% error with 99.9% energy threshold

**When to use MOR:**
- Large models (100+ particles)
- Repeated simulations (control, optimization)
- Real-time applications

**Limitations:**
- Training data must cover operating range
- Not suitable for topology changes
- Collision handling in reduced space is challenging

**Next:** [05_sdf.ipynb](05_sdf.ipynb) - Add SDF terrain collision for complex environments!